# Imports

In [1]:
from May31_Frank2D_CPU import *

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from constants import rad_to_arcsec, deg_to_rad
from scipy.stats import binned_statistic
from matplotlib.colors import LogNorm

# frank1d utilities
from frank.geometry import SourceGeometry
from frank.radial_fitters import FrankFitter
from frank.utilities import UVDataBinner
import matplotlib.colors as colors

class Plot(object):
    def __init__(self, Frank2D, Geometry):
        self._frank2d = Frank2D
        self._Geometry = Geometry
        self._u_gridded, self._v_gridded = self._frank2d._gridded_data['u'], self._frank2d._gridded_data['v']
        self._vis_gridded, self._weights_gridded = self._frank2d._gridded_data['vis'], self._frank2d._gridded_data['weights']
        self._Nx = Frank2D._Nx
        self._Ny = Frank2D._Ny

        
    def intensity_model(self, title="Frank2D intensity model", deproject = False, zoom = None, fig_size = 6, vmax = 4e10):
        frank2d = self._frank2d
        I = frank2d.sol_intensity
        x_ = frank2d._FT._Xn * rad_to_arcsec
        y_ = frank2d._FT._Yn * rad_to_arcsec
        Nx, Ny = self._Nx, self._Ny
        x, y = x_, y_

        # Deprojection.
        inc_r = self._Geometry._inc * deg_to_rad
        pa_r = self._Geometry._pa * deg_to_rad

        cos_i = np.cos(inc_r)
        cos_pa, sin_pa = np.cos(pa_r), np.sin(pa_r)
        
        if deproject:
            x = (x_ * cos_pa + y_ * sin_pa) / cos_i
            y = (x_ * -sin_pa + y_ * cos_pa)

        plt.figure(figsize=(fig_size, fig_size))

        X = x.reshape(Ny, Nx)
        Y = y.reshape(Ny, Nx)
        I = I.reshape(Ny, Nx)

        I_flip = np.fliplr(I)
        X_flip = -np.fliplr(X)

        norm = colors.PowerNorm(gamma=0.45)
        plot = plt.pcolormesh(X_flip, Y, I_flip,
                              cmap='magma',norm=norm)
        plt.gca().invert_xaxis()
        cmap = plt.colorbar(plot, shrink=0.8)
        cmap.set_label(r'I [Jy $sr^{-1}$]', size=15)

        plt.title(title)
        plt.xlabel("dRa ['']")
        plt.ylabel("dDec ['']")

        plt.gca().set_aspect(1)  


        xlim = plt.xlim()
        ylim = plt.ylim()
        
        lims = [xlim[1], xlim[0]]

        if zoom is not None:
            plt.xlim(zoom, -zoom)
            plt.ylim(-zoom, zoom)

        """
        plt.text(
            lims[0] + 0.9 * (lims[1] - lims[0]),  
            lims[0] + 0.05 * (lims[1] - lims[0]),  
            r'  $'+ str(Nx) + '^{2}$ pixels ',
            bbox={'facecolor': 'white', 'pad': 4, 'alpha': 0.8}
            )
        """

        plt.show()


    def visibility_model(self, title="Frank2D visibility model", deproject = False):
            frank2d = self._frank2d
            Nx, Ny = self._Nx, self._Ny

            vis_model = frank2d.sol_visibility.reshape(Nx, Ny)
            
            u, v = frank2d._FT._Un, frank2d._FT._Vn

            u_shifted, v_shifted = np.fft.fftshift(u.reshape(Nx, Ny)), np.fft.fftshift(v.reshape(Nx, Ny))
            vis_shifted = np.fft.fftshift(vis_model)

            if deproject: 
                u_shifted, v_shifted, _ = self._Geometry.deproject(u_shifted.flatten(), v_shifted.flatten())

            plt.pcolormesh(v_shifted,
                           u_shifted,
                           np.log(np.abs(vis_shifted)),
                           cmap="viridis", vmin=-12, vmax=-2)
            plt.xlabel(r'u [ $\lambda$]')
            plt.ylabel(r'v [ $\lambda$]')
            plt.gca().set_aspect('equal') 
            cmap = plt.colorbar(shrink=0.8)
            cmap.set_label(r'V [Jy]', size=15)
            plt.title(r'log|$Vis_{model}$|')
            plt.show()


    def visibility_gridded_input(self, title="Frank2D gridded input", deproject = False):
        frank2d = self._frank2d
        Nx, Ny = self._Nx, self._Ny
        
        u_gridded, v_gridded = self._u_gridded, self._v_gridded
        vis_gridded, weights_gridded = self._vis_gridded, self._weights_gridded
        
        u_shifted, v_shifted = np.fft.fftshift(u_gridded.reshape(Nx, Ny)), np.fft.fftshift(v_gridded.reshape(Nx, Ny))
        vis_shifted = np.fft.fftshift(vis_gridded.reshape(Nx, Ny))

        if deproject: 
            u_shifted, v_shifted, _ = self._Geometry.deproject(u_shifted.flatten(), v_shifted.flatten())

        
        plt.pcolormesh(v_shifted,
                       u_shifted,
                       np.log(np.abs(vis_shifted)),
                       cmap="viridis", vmin=-12, vmax=-2)
        plt.xlabel(r'u [1e6 $\lambda$]')
        plt.ylabel(r'v [1e6 $\lambda$]')
        plt.title(r'log |$Vis_{Input}$|')
        cmap = plt.colorbar()
        cmap.set_label(r'V [Jy]', size=15)


    def  get_vis_profile(self, n_bins, range =  None, weighted = False, deprojected = False):
        u = self._frank2d._FT._Un
        v = self._frank2d._FT._Vn
    
        inc_r = self._frank2d._Geometry._inc*deg_to_rad
        pa_r = self._frank2d._Geometry._pa*deg_to_rad

        if deprojected:
            cos_i = np.cos(inc_r)
            cos_pa, sin_pa = np.cos(pa_r), np.sin(pa_r)
        
            u_ = u * cos_pa - v * sin_pa
            v_d = u * sin_pa + v * cos_pa
            u_d = u_ * cos_i
            u, v = u_d, v_d
        
        q = np.hypot(u, v)
        
        Vis = self._frank2d.sol_visibility
        q = q.flatten()
        Vis = Vis.flatten()
        
        weights_gridded = self._frank2d._gridded_data['weights']
        Vis_binned, bin_edges, _ = binned_statistic(q, Vis, 'mean', bins = n_bins, range = range)
        q_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        
        if weighted:
            if range is None:
                Vis_Weights_binned, bin_edges, _ = binned_statistic(q, Vis*weights_gridded, 'sum', bins = n_bins)
                Weights_binned, bin_edges, _ = binned_statistic(q, weights_gridded, 'sum', bins = n_bins)
            else:
                Vis_Weights_binned, bin_edges, _ = binned_statistic(q, Vis*weights_gridded, 'sum', bins = n_bins, range = range)
                Weights_binned, bin_edges, _ = binned_statistic(q, weights_gridded, 'sum', bins = n_bins, range = range)
                
            Vis_Weights_binned = np.nan_to_num(Vis_Weights_binned, nan=0)
            return q_centers, Vis_Weights_binned/Weights_binned
    
        return q_centers, Vis_binned
        
    def frank1d(self, u, v, Vis, Weights, alpha = 1.3, w_smooth = 1e-3, n_pts = 300):
        Rout = self._frank2d._Rmax*rad_to_arcsec
        geom = self._Geometry
        inc, pa, dra, ddec = geom._inc, geom._pa, geom._dra, geom._ddec
        
        geom_f1d = SourceGeometry(inc= inc, PA= pa, dRA= dra, dDec= ddec)
        FF = FrankFitter(Rout, n_pts, geom_f1d, alpha = alpha, weights_smooth = w_smooth)
        sol = FF.fit(u, v, Vis, Weights)
        return sol
        
    def compare_vis_profile_frank1d(self, u, v, Vis, Weights):
        #  Plot variables.
        cs, ms = ['#a4a4a4', 'k'], ['.', 'x']
        bin_widths = [1e3, 1e5]

        u_deproj, v_deproj, _ = self._Geometry.deproject(u, v)

        
        baselines = np.hypot(u_deproj, v_deproj)         
        grid = np.logspace(np.log10(min(baselines.min(), baselines[0])),
                                   np.log10(max(baselines.max(), baselines[-1])),
                                   10**4)
        
        
        plt.figure(figsize=(14,5))
        
        # Raw visibilities -------------------------------------------------------------------
        binned_vis = UVDataBinner(baselines, Vis, Weights, bin_widths[0])
        plt.plot(binned_vis.uv, np.abs(binned_vis.V), c=cs[0],
                     marker=ms[0], ls='None', 
                     label=r'Obs., {:.0f} k$\lambda$ bins'.format(bin_widths[0]/1e3))
        
        binned_vis = UVDataBinner(baselines, Vis, Weights, bin_widths[1])
        plt.plot(binned_vis.uv, np.abs(binned_vis.V), c=cs[1],
                     marker=ms[1], ls='None', 
                     label=r'Obs., {:.0f} k$\lambda$ bins'.format(bin_widths[1]/1e3))

        # Frank1D -----------------------------------------------------------------------------
        Nbins = 1000
        sol = self.frank1d(u, v, Vis, Weights)
        baselines_edges = np.geomspace(sol.q[0], sol.q[-1], Nbins + 1 )
        baselines_centers = np.sqrt(baselines_edges[:-1] * baselines_edges[1:]) 
        range = (sol.q[0], sol.q[-1])
        
        vis_fit_1d = sol.predict_deprojected(baselines_centers)
        plt.plot(baselines_centers, np.abs(vis_fit_1d), color = "red", label = r'frank1d', ls ='--')

        # Frank2D (deprojected)------------------------------------------------------------------
        q, Vis_model = self.get_vis_profile(baselines_edges, range, weighted = True, deprojected = True)
        plt.plot(baselines_centers, np.abs(Vis_model), label = f'frank2d', color = 'blue')

        plt.xlabel(r'baseline [$\lambda$]')
        plt.xscale('log')
        plt.yscale('log')
        plt.ylim(1e-5, 10)
        plt.xlim(2e5, 6e6)
        plt.ylabel('|V| [Jy]', size = 10)
        plt.title(r'$Visibility_{Model}$ N = ' + str(self._Nx))
        plt.legend(fontsize= 10, loc = 'best')
        plt.show()

# Frank2D

In [3]:
def frank2d_cpu(disk_name, filename, disk_geometry, Rout, N = 100, maxiter=20000):
    print("Running: " + disk_name +" ...." + " frank2d model with " + str(N)+ 'x'+str(N)+ " points" )

    inc = disk_geometry['inc']
    pa = disk_geometry['pa']
    dra = disk_geometry['dra']
    ddec = disk_geometry['ddec']
    
    # UVtable
    dir = "../../../data/"
    data_file = dir + filename
    
     # load data
    data = np.load(data_file)
    u, v, Re, Imag, Weights = data['u'], data['v'], data['Re'], data['Im'], data['w']
    Vis = Re + Imag*1j
    
    #idx = (np.abs(u) < 2e6) & (np.abs(v) < 2e6)
    #u, v, Vis, Weights =  u[idx], v[idx], Vis[idx], Weights[idx]
    u, v, Vis, Weights =  u, v, Vis, Weights

    
    geom = Geometry(inc, pa, dra, ddec)
    frank2d = Frank2D(N, Rout, geom)
    
    start_time = time.time()
    
    frank2d.preprocess_vis(u, v, Vis, Weights, hermitian= True)
    u_gridded, v_gridded, vis_gridded, weights_gridded = frank2d._gridded_data['u'], frank2d._gridded_data['v'], frank2d._gridded_data['vis'], frank2d._gridded_data['weights']
    sol = Frank2D_optimized(N, u_gridded, v_gridded, vis_gridded, weights_gridded, maxiter = maxiter)
    frank2d.sol_visibility = sol
    frank2d.fft()
    
    end_time = time.time()
    execution_time = end_time - start_time
    print(f'  ------------------------------------------> TOTAL ALGORITHM TIME = {execution_time/60 :.2f}  min | {execution_time: .2f} seconds')

    return u_gridded, v_gridded, sol, frank2d, geom, [u, v, Vis, Weights]

# AS209 

In [ ]:
u_gridded_AS209, v_gridded_AS209, Vis_model_AS209, FrankObj_AS209, GeomObj_AS209, input_frank_AS209 = frank2d_cpu(
                                            'AS209 1mm',
                                            'uvtables/uvtable_AS209_continuum.npz',
                                            {'inc': 34.97, 'pa': 85.76, 'dra':1.9e-3, 'ddec':-2.5e-3 },
                                            3,
                                            300
                                            )

Running: AS209 1mm .... frank2d model with 300x300 points


/Users/mariajmelladot/Desktop/Frank2D/6_Frank2D_Oficial/frank2d/preprocess_vis.py:46: RuntimeWarning: invalid value encountered in divide
  vis_gridded_matrix =  vis_weights_sum_bin/weights_gridded_matrix


Enforcing Hermitian symmetry...
  --> time = 0.37  min |  22.05 seconds
Setting gridded data...
RUNNING WITH  {'m': -2.5, 'c': 11.5, 'l': 100000.0}
------> Creating linear operators


In [ ]:
# Plots.
Plot_AS209 = Plot(FrankObj_AS209, GeomObj_AS209)

In [ ]:
u_AS209, v_AS209, Vis_AS209, Weights_AS209 = input_frank_AS209
Rout_AS209 = 3
#Plot_AS209.compare_vis_profile_frank1d(Rout_AS209, u_AS209, v_AS209, Vis_AS209, Weights_AS209)

In [ ]:
Plot_AS209.intensity_model(vmax = 4e10)
Plot_AS209.visibility_model()
Plot_AS209.visibility_gridded_input()

In [ ]:
import pickle
name = "../../../data/geom_AS209"
obj = GeomObj_AS209
with open(name + ".pickle", "wb") as f:
    pickle.dump(obj, f)

# HD163006 

In [ ]:
u_gridded_HD163006, v_gridded_HD163006, Vis_model_HD163006, FrankObj_HD163006, GeomObj_HD163006, input_frank_HD163006 = frank2d_cpu(
                                                'HD143006 1mm',
                                                'uvtables/non_axisymmetric/uvtable_HD143006_continuum.npz', 
                                                {'inc': 18.6, 'pa': 169, 'dra':-5.9e-3, 'ddec':21.7e-3 },
                                                0.518*2,
                                                N = 200
                                                )

In [ ]:
# Plots.
Plot_HD163006 = Plot(FrankObj_HD163006, GeomObj_HD163006)
Plot_HD163006.intensity_model(vmax = 5e10)
Plot_HD163006.visibility_model()
Plot_HD163006.visibility_gridded_input()

# Elias27

In [ ]:
u_gridded_Elias27, v_gridded_Elias27, Vis_model_Elias27, FrankObj_Elias27, GeomObj_Elias27, input_frank_Elias27 = frank2d_cpu(
                                            'Elias27 1mm',
                                            'uvtables/non_axisymmetric/uvtable_Elias27_continuum.npz',
                                            {'inc': 56.2, 'pa': 118.8, 'dra':-5e-3, 'ddec':-8e-3 },
                                            1.88*2,
                                            300
                                            )

In [ ]:
# Plots.
Plot_Elias27 = Plot(FrankObj_Elias27, GeomObj_Elias27)

In [ ]:
import pickle
name = "frank2d_Elias27_300"
obj = FrankObj_Elias27
with open(name + ".pickle", "wb") as f:
    pickle.dump(obj, f)

In [ ]:
Plot_Elias27.intensity_model(vmax = 1e5, title = r'Frank2D: Elias27 - $I_{model}$')

In [ ]:
Plot_Elias27.visibility_model()

In [ ]:
Plot_Elias27.visibility_gridded_input()

In [ ]:
Plot_Elias27.get_vis_profile(100)

# IMLup

In [ ]:
u_gridded_IMLup, v_gridded_IMLup, Vis_model_IMLup, FrankObj_IMLup, GeomObj_IMLup, input_frank_IMLup =  frank2d_cpu(
                                            'IMLup 1mm',
                                            'uvtables/non_axisymmetric/uvtable_IMLup_continuum.npz',
                                            {'inc': 47.5, 'pa': 144.5, 'dra':-1.5e-3, 'ddec':1e-3 },
                                            1.71*2,
                                            N = 100, 
                                            maxiter = 40000
                                            )

In [ ]:
# Plots.
Plot_IMLup = Plot(FrankObj_IMLup, GeomObj_IMLup)
Plot_IMLup.intensity_model()
Plot_IMLup.visibility_model()
Plot_IMLup.visibility_gridded_input()

# Elias24

In [ ]:
u_gridded_Elias24, v_gridded_Elias24, Vis_model_Elias24, FrankObj_Elias24, GeomObj_Elias24, input_frank_Elias24 = frank2d_cpu(
                                            'Elias24 1mm',
                                            'uvtables/non_axisymmetric/uvtable_Elias24_continuum.npz',
                                            {'inc': 29, 'pa': 45.7, 'dra':110.8e-3, 'ddec':-386.8e-3 },
                                            1.03*2,
                                            100
                                            )

In [ ]:
# Plots.
Plot_Elias24 = Plot(FrankObj_Elias24, GeomObj_Elias24)
Plot_Elias24.intensity_model()
Plot_Elias24.visibility_model()
Plot_Elias24.visibility_gridded_input()

# HD163296

In [ ]:
u_gridded_HD163296, v_gridded_HD163296, Vis_model_HD163296, FrankObj_HD163296, GeomObj_HD163296, input_frank_HD163296 = frank2d_cpu(
                                            'HD163296 1mm',
                                            'uvtables/non_axisymmetric/uvtable_HD163296_continuum.npz',
                                            {'inc':46.7, 'pa':133.33, 'dra':-2.8e-3, 'ddec':7.7e-3},
                                            1.7*2,
                                            100
                                            )

In [ ]:
# Plots.
Plot_HD163296 = Plot(FrankObj_HD163296, GeomObj_HD163296)
Plot_HD163296.intensity_model()
Plot_HD163296.visibility_model()
Plot_HD163296.visibility_gridded_input()